In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp

# 1. Khởi tạo Spark Session
spark = SparkSession.builder.appName("Bronze_Ingestion").getOrCreate()

# 2. Đọc luồng dữ liệu real-time từ Kafka (Redpanda)
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "redpanda:9092") \
    .option("subscribe", "yellow_trips") \
    .option("startingOffsets", "earliest") \
    .load()

# 3. Biến đổi dữ liệu theo đúng chuẩn Tầng Bronze:
# - Chuyển cột value thành chuỗi JSON thô (json_str)
# - Bổ sung các trường audit (ingested_at, offset, partition)
# - KHÔNG dùng from_json hay ép schema ở tầng này để tránh làm văng luồng nếu data lỗi
df_bronze = df_kafka.select(
    col("offset"),
    col("partition"),
    col("value").cast("string").alias("json_str"),
    current_timestamp().alias("ingested_at")
)

# 4. Ghi luồng xuống bảng Iceberg ở Tầng Bronze (Chế độ Append-only)
query = df_bronze.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .option("checkpointLocation", "/data/checkpoints/bronze_trips") \
    .toTable("bronze.taxi_trips_raw")

# Kích hoạt luồng chạy liên tục
query.awaitTermination()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/21 09:20:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/21 09:20:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


AnalysisException: [SCHEMA_NOT_FOUND] The schema `bronze` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a catalog, verify the current_schema() output, or qualify the name with the correct catalog.
To tolerate the error on drop use DROP SCHEMA IF EXISTS.